# Qwen2.5-VL Custom Supervised Fine-Tuning (SFT) Training (Kaggle 2xT4 Multi-GPU)

Executes minimal custom SFT training on Qwen2.5-VL using target reasoning solutions with automatic model parallelism across 2xT4 GPUs.

In [ ]:
# Cell 1: Check GPU hardware and install sm_60 compatible PyTorch stack if Tesla P100 is assigned
import os, subprocess, sys, torch

print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.12 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print('Packages successfully configured.')

In [ ]:
# Cell 2: Checkout repository and execute custom SFT trainer
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/tmp/prm_project')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/prm_project.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

# Run full custom SFT training!
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'train_sft.py',
    '--dataset-path', 'experiments/001_500_reasoning/data/sft_samples.jsonl',
    '--output-dir', '/kaggle/working/qwen_vl_sft_adapter',
    '--epochs', '3',
    '--batch-size', '1',
    '--lr', '2e-5'
]
subprocess.run(cmd, env=env, check=True)

In [ ]:
# Cell 3: Validate output artifacts
out_dir = Path('/kaggle/working/qwen_vl_sft_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'SFT Adapter directory {out_dir} contents: {files}')